**What are `pmdarima` and `statsmodels`?**

- `pmdarima`: A Python library for time series analysis, focused on automating ARIMA model selection (`auto_arima`). It helps find the best parameters for forecasting without manual tuning.
- `statsmodels`: A comprehensive library for statistical modeling, including SARIMAX (Seasonal ARIMA with exogenous variables), which is used to fit and forecast time series data.

**Why use them?**

- `pmdarima` quickly identifies optimal ARIMA/SARIMA parameters, saving time and improving accuracy.
- `statsmodels` provides robust tools to fit, evaluate, and forecast with the chosen model, supporting advanced statistical analysis and diagnostics.

In [0]:
%pip install pmdarima statsmodels

In [0]:
import mlflow
import mlflow.pyfunc
import numpy as np
import pandas as pd
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import pickle, os

# Cell 1: Load and split
pdf = spark.table("weather_silver").toPandas().sort_values("date").set_index("date")
ts = pdf["avg_temp"].dropna()

train_size = int(len(ts) * 0.8)
train, test = ts[:train_size], ts[train_size:]



## 🔍 How `auto_arima` Parameters Are Determined

### **You Manually Decide** (based on EDA):

| Parameter | Value | How Determined |
|-----------|-------|----------------|
| `seasonal=True` | True | From **EDA notebook** - saw weekly patterns in time series plot |
| `m=7` | 7 | From **ACF/PACF plots** - repeating patterns every 7 days (weekly seasonality) |
| `d=1` | 1 | From **ADF test** - series was non-stationary, needs 1st differencing |
| `D=1` | 1 | From **visual inspection** - seasonal trend exists, needs seasonal differencing |
| `max_p`, `max_q` | 3 | **Computational constraint** - limit search space (ACF/PACF suggest p,q < 3) |
| `max_P`, `max_Q` | 2 | **Computational constraint** - seasonal components usually smaller |
| `information_criterion` | "aic" | **Standard practice** - AIC balances fit vs complexity |

### **`auto_arima` Automatically Searches**:

`auto_arima` tries **all combinations** of:
- `p` = 0, 1, 2, 3 (AR order)
- `q` = 0, 1, 2, 3 (MA order)
- `P` = 0, 1, 2 (Seasonal AR)
- `Q` = 0, 1, 2 (Seasonal MA)

That's **4 × 4 × 3 × 3 = 144 possible models**!

It compares their AIC scores and returns the **best** combination.

---

### 💡 How to Determine These Yourself:

**Step 1: Seasonality** (`seasonal`, `m`)
- Plot your time series
- Look for repeating patterns
- Daily data with weekly pattern → `m=7`
- Monthly data with yearly pattern → `m=12`

**Step 2: Differencing** (`d`, `D`)
- Run **ADF test** (in EDA notebook)
- If p-value > 0.05 → non-stationary → set `d=1`
- If seasonal trend visible → set `D=1`
- OR let `auto_arima` find it by removing `d` and `D` parameters

**Step 3: Search Limits** (`max_p`, `max_q`, `max_P`, `max_Q`)
- Check **ACF/PACF plots** (in EDA notebook)
- If ACF cuts off at lag 2 → try `max_q=2`
- If PACF cuts off at lag 3 → try `max_p=3`
- **Higher values = slower but more thorough search**

**Step 4: Let it Run**
- With `trace=True`, you'll see every model tested
- Best model (lowest AIC) is selected automatically

---

### 🎯 Your Current Setup:

From your EDA, you determined:
1. ✅ Weekly seasonality exists (`m=7`)
2. ✅ Needs differencing (`d=1`, `D=1`)
3. ✅ ACF/PACF suggest p,q ≤ 3

So `auto_arima` searches 144 models and found:
- **Best order**: `(2, 1, 1)` - meaning p=2, d=1, q=1
- **Seasonal order**: `(0, 1, 2, 7)` - meaning P=0, D=1, Q=2, m=7

This is the optimal SARIMA configuration for your data!

In [0]:
# Cell 2: auto_arima to find best order
print("Running auto_arima — this takes ~2-3 minutes...")
auto_model = auto_arima(
    train,
    seasonal=True, m=7,        # m=7 → weekly seasonality on daily data
    d=1, D=1,
    max_p=3, max_q=3,
    max_P=2, max_Q=2,
    information_criterion="aic",
    trace=True,
    error_action="ignore",
    suppress_warnings=True
)
print(f"Best order: {auto_model.order}  Seasonal order: {auto_model.seasonal_order}")
print(f"Best AIC: {auto_model.aic():.2f}")




### Line-by-Line Explanation of the Cell

python
print("Running auto_arima — this takes ~2-3 minutes...")

- Prints a message to inform the user that the `auto_arima` process is starting and may take a few minutes.

python
auto_model = auto_arima(
    train,
    seasonal=True, m=7,        # m=7 → weekly seasonality on daily data
    d=1, D=1,
    max_p=3, max_q=3,
    max_P=2, max_Q=2,
    information_criterion="aic",
    trace=True,
    error_action="ignore",
    suppress_warnings=True
)

- Runs `auto_arima` on the training data to automatically find the best SARIMA model parameters.
    - `seasonal=True, m=7`: Enables weekly seasonality (7 days).
    - `d=1, D=1`: Sets differencing for trend and seasonality.
    - `max_p=3, max_q=3`: Limits AR and MA orders to 3.
    - `max_P=2, max_Q=2`: Limits seasonal AR and MA orders to 2.
    - `information_criterion="aic"`: Uses AIC to select the best model.
    - `trace=True`: Prints progress for each model tested.
    - `error_action="ignore"`: Ignores errors during model fitting.
    - `suppress_warnings=True`: Suppresses warnings.

python
print(f"Best order: {auto_model.order}  Seasonal order: {auto_model.seasonal_order}")

- Prints the best non-seasonal and seasonal orders found by `auto_arima`.

python
print(f"Best AIC: {auto_model.aic():.2f}")

- Prints the AIC value of the best model, rounded to two decimal places.

In [0]:
# Cell 3: MLflow experiment — log everything
mlflow.set_experiment("/Users/santhoshnagendrarajan@gmail.com/weather-sarima")

def evaluate_and_log(order, seasonal_order, run_name):
    with mlflow.start_run(run_name=run_name):
        model = SARIMAX(train, order=order, seasonal_order=seasonal_order,
                        enforce_stationarity=False, enforce_invertibility=False)
        result = model.fit(disp=False)

        forecast = result.forecast(steps=len(test))
        rmse = np.sqrt(mean_squared_error(test, forecast))
        mae  = mean_absolute_error(test, forecast)
        mape = np.mean(np.abs((test.values - forecast.values) / test.values)) * 100

        # Log params
        mlflow.log_param("p", order[0]); mlflow.log_param("d", order[1]); mlflow.log_param("q", order[2])
        mlflow.log_param("P", seasonal_order[0]); mlflow.log_param("S", seasonal_order[3])
        mlflow.log_param("aic", round(result.aic, 2))

        # Log metrics
        mlflow.log_metric("rmse", round(rmse, 4))
        mlflow.log_metric("mae",  round(mae, 4))
        mlflow.log_metric("mape", round(mape, 4))

        # Save forecast plot as artifact
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(train[-60:], label="Train (last 60 days)", color="#1D9E75")
        ax.plot(test, label="Actual", color="#378ADD")
        ax.plot(test.index, forecast, label="Forecast", color="#D85A30", linestyle="--")
        ax.fill_between(test.index,
                        result.get_forecast(steps=len(test)).conf_int().iloc[:,0],
                        result.get_forecast(steps=len(test)).conf_int().iloc[:,1],
                        alpha=0.2, color="#D85A30")
        ax.legend(); ax.set_title(f"{run_name} | RMSE={rmse:.2f}")
        fig.savefig("/tmp/forecast_plot.png", dpi=150)
        mlflow.log_artifact("/tmp/forecast_plot.png")

        # Save model pickle as artifact
        with open("/tmp/sarima_model.pkl", "wb") as f:
            pickle.dump(result, f)
        mlflow.log_artifact("/tmp/sarima_model.pkl")

        print(f"{run_name} → RMSE: {rmse:.3f}  MAE: {mae:.3f}  MAPE: {mape:.2f}%")
        return rmse, result


## 📝 Line-by-Line Explanation of Cell 7

### **Overview**
This cell defines a function `evaluate_and_log()` that trains a SARIMA model, evaluates it, and logs everything to MLflow for experiment tracking.

---

### **Line 1-2: Set MLflow Experiment**
```python
mlflow.set_experiment("/Users/santhoshnagendrarajan@gmail.com/weather-sarima")
```
**What it does:** Creates or connects to an MLflow experiment named `weather-sarima`.
- All subsequent runs will be logged under this experiment
- You can view all runs at `/ml/experiments/{experiment_id}` in Databricks UI

---

### **Line 3: Define Function**
```python
def evaluate_and_log(order, seasonal_order, run_name):
```
**Parameters:**
- `order`: Tuple `(p, d, q)` for non-seasonal SARIMA parameters
- `seasonal_order`: Tuple `(P, D, Q, m)` for seasonal parameters
- `run_name`: Human-readable name for this experiment run (e.g., "auto_arima_best")

---

### **Line 4: Start MLflow Run**
```python
    with mlflow.start_run(run_name=run_name):
```
**What it does:** Creates a new experiment run to track all parameters, metrics, and artifacts.
- Uses context manager (`with`) to automatically close the run when done
- All logging happens inside this block

---

### **Lines 5-7: Train SARIMA Model**
```python
        model = SARIMAX(train, order=order, seasonal_order=seasonal_order,
                        enforce_stationarity=False, enforce_invertibility=False)
        result = model.fit(disp=False)
```
**What it does:**
1. **Line 5:** Create SARIMAX model object with specified parameters
   - `train`: Your training data (584 days of temperature)
   - `order=(p, d, q)`: Non-seasonal parameters
   - `seasonal_order=(P, D, Q, m)`: Seasonal parameters
2. **Line 6:** Optimization flags
   - `enforce_stationarity=False`: Don't force AR parameters to be stationary (more flexible)
   - `enforce_invertibility=False`: Don't force MA parameters to be invertible (more flexible)
3. **Line 7:** Fit the model to training data
   - `disp=False`: Don't print convergence messages
   - Returns `result` object with fitted parameters

---

### **Lines 9-11: Generate Forecast & Calculate Metrics**
```python
        forecast = result.forecast(steps=len(test))
        rmse = np.sqrt(mean_squared_error(test, forecast))
        mae  = mean_absolute_error(test, forecast)
        mape = np.mean(np.abs((test.values - forecast.values) / test.values)) * 100
```
**What it does:**
1. **Line 9:** Generate forecast for test period (146 days ahead)
2. **Line 10:** Calculate RMSE (Root Mean Squared Error)
   - Penalizes large errors more than small ones
   - In same units as temperature (°C)
3. **Line 11:** Calculate MAE (Mean Absolute Error)
   - Average of absolute errors
   - More robust to outliers than RMSE
4. **Line 12:** Calculate MAPE (Mean Absolute Percentage Error)
   - Error as percentage of actual values
   - Easy to interpret: "forecast is off by X%"

---

### **Lines 13-15: Log Parameters to MLflow**
```python
        mlflow.log_param("p", order[0]); mlflow.log_param("d", order[1]); mlflow.log_param("q", order[2])
        mlflow.log_param("P", seasonal_order[0]); mlflow.log_param("S", seasonal_order[3])
        mlflow.log_param("aic", round(result.aic, 2))
```
**What it does:** Log model hyperparameters for tracking and comparison
- **Line 13:** Log non-seasonal orders (`p`, `d`, `q`)
- **Line 14:** Log seasonal AR order (`P`) and period (`S` = `m`)
- **Line 15:** Log AIC (Akaike Information Criterion) for model quality

**Why log parameters?** So you can compare which configurations work best.

---

### **Lines 17-20: Log Metrics to MLflow**
```python
        mlflow.log_metric("rmse", round(rmse, 4))
        mlflow.log_metric("mae",  round(mae, 4))
        mlflow.log_metric("mape", round(mape, 4))
```
**What it does:** Log performance metrics for model comparison
- MLflow tracks these as the primary success criteria
- You can sort runs by these metrics (e.g., "find run with lowest RMSE")
- Rounded to 4 decimal places for readability

---

### **Lines 22-34: Create & Save Forecast Plot**
```python
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(train[-60:], label="Train (last 60 days)", color="#1D9E75")
        ax.plot(test, label="Actual", color="#378ADD")
        ax.plot(test.index, forecast, label="Forecast", color="#D85A30", linestyle="--")
        ax.fill_between(test.index,
                        result.get_forecast(steps=len(test)).conf_int().iloc[:,0],
                        result.get_forecast(steps=len(test)).conf_int().iloc[:,1],
                        alpha=0.2, color="#D85A30")
        ax.legend(); ax.set_title(f"{run_name} | RMSE={rmse:.2f}")
        fig.savefig("/tmp/forecast_plot.png", dpi=150)
        mlflow.log_artifact("/tmp/forecast_plot.png")
```
**What it does:** Create a visualization showing model performance

**Line-by-line:**
- **Line 22:** Create matplotlib figure (12" wide, 4" tall)
- **Line 23:** Plot last 60 days of training data (green line)
- **Line 24:** Plot actual test values (blue line)
- **Line 25:** Plot forecasted values (orange dashed line)
- **Lines 26-29:** Add shaded confidence interval band
  - `get_forecast().conf_int()`: Get 95% confidence bounds
  - `.iloc[:,0]`: Lower bound
  - `.iloc[:,1]`: Upper bound
  - `alpha=0.2`: 20% transparency (light orange shade)
- **Line 30:** Add legend and title with RMSE
- **Line 31:** Save plot to `/tmp/forecast_plot.png` at 150 DPI
- **Line 32:** Upload plot to MLflow as artifact (viewable in UI)

---

### **Lines 34-37: Save Model as Artifact**
```python
        with open("/tmp/sarima_model.pkl", "wb") as f:
            pickle.dump(result, f)
        mlflow.log_artifact("/tmp/sarima_model.pkl")
```
**What it does:** Save the trained model for later use
- **Line 34:** Open file in write-binary mode
- **Line 35:** Serialize fitted model using pickle
  - Saves all model parameters and state
  - Can be loaded later for predictions
- **Line 36:** Upload model to MLflow artifact store

**Note:** This logs as an artifact, not an MLflow model. To use the model registry, you'd need `mlflow.statsmodels.log_model()` instead.

---

### **Lines 38-39: Print Summary & Return**
```python
        print(f"{run_name} → RMSE: {rmse:.3f}  MAE: {mae:.3f}  MAPE: {mape:.2f}%")
        return rmse, result
```
**What it does:**
- **Line 38:** Print immediate feedback showing metrics
  - Example output: `auto_arima_best → RMSE: 4.372  MAE: 3.512  MAPE: 11.89%`
- **Line 39:** Return RMSE and fitted model for further use

---

## 🎯 Summary: What This Function Does

1. **Takes in:** SARIMA parameters and a run name
2. **Trains:** A SARIMA model on your training data
3. **Evaluates:** Generates forecasts and calculates RMSE, MAE, MAPE
4. **Logs to MLflow:**
   - Parameters: p, d, q, P, S, AIC
   - Metrics: RMSE, MAE, MAPE
   - Artifacts: Forecast plot PNG, Pickled model
5. **Returns:** RMSE and fitted model

---

## 💡 Why This Pattern?

**Reproducibility:** Everything is tracked - you can recreate any experiment

**Comparison:** MLflow UI lets you compare all runs side-by-side

**Deployment:** Best model can be loaded and deployed to production

**Auditing:** Full lineage from data → model → predictions

In [0]:
# Cell 4: Run 3 experiments to compare
best_order = auto_model.order
best_seasonal = auto_model.seasonal_order

evaluate_and_log(best_order,       best_seasonal,          "auto_arima_best")
evaluate_and_log((1, 1, 1),        (1, 1, 0, 7),           "baseline_simple")
evaluate_and_log((best_order[0],1,best_order[2]+1), best_seasonal, "tuned_q_plus1")